# Manifold Analysis — Intrinsic Dimensionality of FM Representations

For every foundation model on `nitrox639/platonic-embeddings` (EEG and vision),
this notebook characterises the *shape* of its representational manifold at every layer:

1. **Linear ID** via PCA **participation ratio** — $(\sum\lambda)^2 / \sum\lambda^2$, a continuous, scale-free effective dimension.
2. **Non-linear ID** via **TwoNN** (Facco et al. 2017) — slope of $-\log(1-F(\mu))$ vs $\log\mu$ on local 2nd/1st-NN ratios.

Both are plotted against **relative depth** in three size tiers (small / mid / large), one curve per FM family.

The final section uses **per-window local TwoNN ID** at each max-size FM's last layer to surface
video chunks at the extremes — windows in *thin* (predictable) vs *fat* (diverse) regions of the manifold —
a manifold-geometry counterpart to the k-NN-distance bridge in `temporal_distance_analysis.ipynb`.


In [ ]:
from __future__ import annotations
import os, io, base64, functools
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from huggingface_hub import hf_hub_download

EMBED_REPO   = 'nitrox639/platonic-embeddings'
VIDEO_REPO   = 'Fudan-fMRI/CineBrain'
K_LOCAL      = 20
GIF_SIZE     = 160
MAX_CLIP_IDX = 2699
CLIP_SECONDS = 4.0
SOURCE_FPS   = 8
SEASON_END_S = (MAX_CLIP_IDX + 1) * CLIP_SECONDS

for _tp in (Path('tokens/hf_token.txt'), Path('hf_token.txt'),
            Path.home() / 'hf_token.txt',
            Path('../tokens/hf_token.txt'),
            Path('../src/extraction_scripts/hf_token.txt')):
    if _tp.exists():
        os.environ['HF_TOKEN'] = _tp.read_text().strip()
        break
HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'No HF token found'

def _fetch(remote_path):
    local = hf_hub_download(EMBED_REPO, remote_path, repo_type='dataset', token=HF_TOKEN)
    return np.load(local, allow_pickle=True)['embeddings'].astype(np.float32)

def _l2(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

print('Setup complete.')


In [ ]:
# Vision family choice for ID analysis: use femba_luna (2160 windows, 5 s each).
# The manifold-shape of a vision FM is computed over its set of stimuli; the
# EEG-family tag only controls window length / count, not the FM's parameters.
_VIS_FAM = 'femba_luna'

REGISTRY = {
    'femba':       dict(modality='eeg', family='femba_luna',  window_s=5,
                        label='FEMBA',       color='#1f77b4',
                        path=lambda s: f'eeg/femba/femba_{s}_tusl_layerwise.npz',
                        tiers={'small': 'tiny', 'mid': 'base',  'large': 'large'}),
    'luna':        dict(modality='eeg', family='femba_luna',  window_s=5,
                        label='LUNA',        color='#2ca02c',
                        path=lambda s: f'eeg/luna/luna_{s}_layerwise.npz',
                        tiers={'small': 'base', 'mid': 'large', 'large': 'huge'}),
    'neurolm':     dict(modality='eeg', family='neurolm',     window_s=8,
                        label='NeuroLM',     color='#17becf',
                        path=lambda s: f'eeg/neurolm/neurolm_{s}_layerwise.npz',
                        tiers={'small': 'b', 'mid': 'l', 'large': 'xl'}),
    'steegformer': dict(modality='eeg', family='steegformer', window_s=6,
                        label='STEEGFormer', color='#9467bd',
                        path=lambda s: f'eeg/steegformer/steegformer_{s}_layerwise.npz',
                        tiers={'small': 'small', 'mid': 'base', 'large': 'large'}),
    'reve':        dict(modality='eeg', family='reve',        window_s=10,
                        label='REVE',        color='#8c564b',
                        path=lambda s: f'eeg/reve/reve_{s}_layerwise.npz',
                        tiers={'small': 'base', 'large': 'large'}),
    'videomae':    dict(modality='vis', family=_VIS_FAM,      window_s=5,
                        label='VideoMAE',    color='#e07b39',
                        path=lambda s: f'vision/videomae/videomae_{s}__{_VIS_FAM}.npz',
                        tiers={'small': 'base', 'large': 'large'}),
    'dinov2':      dict(modality='vis', family=_VIS_FAM,      window_s=5,
                        label='DINOv2',      color='#d62728',
                        path=lambda s: f'vision/dinov2/dinov2_{s}__{_VIS_FAM}.npz',
                        tiers={'small': 'small', 'mid': 'base', 'large': 'giant'}),
    'vjepa2':      dict(modality='vis', family=_VIS_FAM,      window_s=5,
                        label='V-JEPA 2',    color='#ff7f0e',
                        path=lambda s: f'vision/vjepa2/vjepa2_{s}__{_VIS_FAM}.npz',
                        tiers={'small': 'large', 'mid': 'huge', 'large': 'giant'}),
    'videomae_ft_kinetics': dict(
                        modality='vis', family=_VIS_FAM,      window_s=5,
                        label='VideoMAE-K400ft', color='#bcbd22',
                        path=lambda s: f'vision/videomae_finetuned/videomae_ft_kinetics_{s}__{_VIS_FAM}.npz',
                        tiers={'small': 'base', 'mid': 'large', 'large': 'huge'}),
}

TIERS = ('small', 'mid', 'large')

def iter_tier(tier):
    for fm, cfg in REGISTRY.items():
        if tier in cfg['tiers']:
            yield fm, cfg['tiers'][tier]

print(f'Registry size: {len(REGISTRY)}')
for t in TIERS:
    members = list(iter_tier(t))
    print(f'  {t:5s} ({len(members)}):', ', '.join(f'{f}-{s}' for f, s in members))


In [ ]:
def participation_ratio(X):
    """Linear ID. X: (N, D). Returns (sum lambda)^2 / sum(lambda^2)."""
    Xc = X - X.mean(axis=0, keepdims=True)
    if Xc.shape[1] > Xc.shape[0]:
        G = Xc @ Xc.T
        evals = np.linalg.eigvalsh(G).clip(min=0)
    else:
        C = Xc.T @ Xc
        evals = np.linalg.eigvalsh(C).clip(min=0)
    s = evals.sum()
    return float(s * s / (np.square(evals).sum() + 1e-12))

def twonn_id(X, frac_keep=0.9):
    """Non-linear ID (Facco et al. 2017). X: (N, D), Euclidean kNN.
    Slope of -log(1 - F(mu)) vs log(mu) on the lower frac_keep fraction
    of mu = r2/r1 (drops the top tail per Facco's recipe)."""
    if X.shape[0] < 4:
        return float('nan')
    nn = NearestNeighbors(n_neighbors=3, metric='euclidean').fit(X)
    d, _ = nn.kneighbors(X)
    r1, r2 = d[:, 1], d[:, 2]
    mu = r2 / np.maximum(r1, 1e-12)
    mu = mu[mu > 1.0 + 1e-12]
    if len(mu) < 4:
        return float('nan')
    mu.sort()
    n = max(int(len(mu) * frac_keep), 4)
    mu = mu[:n]
    F = np.arange(1, len(mu) + 1) / (len(mu) + 1)
    x = np.log(mu)
    y = -np.log(1 - F)
    slope, _ = np.polyfit(x, y, 1)
    return float(slope)

print('ID estimators ready.')


In [ ]:
_emb_cache = {}
_id_cache  = {}   # (fm, size) -> dict(L, W, D, rel, pr, twonn)

def _load_emb(fm, size):
    key = (fm, size)
    if key in _emb_cache:
        return _emb_cache[key]
    cfg = REGISTRY[fm]
    path = cfg['path'](size)
    print(f'  fetching {path}', flush=True)
    emb = _fetch(path)
    if cfg['modality'] == 'eeg':
        # (L, W, S, D) -> (L, W, D) subject-mean
        emb = emb.mean(axis=2)
    if emb.ndim == 2:
        emb = emb[None]   # promote (W, D) to (1, W, D) just in case
    _emb_cache[key] = emb
    return emb

def compute_id_for(fm, size):
    key = (fm, size)
    if key in _id_cache:
        return _id_cache[key]
    cfg = REGISTRY[fm]
    emb = _load_emb(fm, size)
    L, W, D = emb.shape
    print(f'  {cfg["label"]}-{size}: L={L}  W={W}  D={D}', flush=True)
    pr  = np.zeros(L)
    tnn = np.zeros(L)
    for l in range(L):
        z = _l2(emb[l])
        pr[l]  = participation_ratio(z)
        tnn[l] = twonn_id(z)
    rel = np.linspace(0.0, 1.0, L) if L > 1 else np.array([0.5])
    _id_cache[key] = dict(L=L, W=W, D=D, rel=rel, pr=pr, twonn=tnn)
    return _id_cache[key]

all_to_compute = []
for tier in TIERS:
    for fm, size in iter_tier(tier):
        if (fm, size) not in all_to_compute:
            all_to_compute.append((fm, size))

print(f'Computing ID for {len(all_to_compute)} (FM, size) combos...\n')
for fm, size in all_to_compute:
    compute_id_for(fm, size)
print('\nDone. Cached IDs for:', list(_id_cache.keys()))


In [ ]:
def _plot_tier_panel(ax, tier, metric_key, ylabel):
    for fm, size in iter_tier(tier):
        cfg = REGISTRY[fm]
        d   = _id_cache[(fm, size)]
        linestyle = '-' if cfg['modality'] == 'eeg' else '--'
        ax.plot(d['rel'], d[metric_key],
                color=cfg['color'], linestyle=linestyle,
                marker='o', markersize=3, linewidth=1.6,
                label=f'{cfg["label"]}-{size}')
    ax.set_xlabel('relative depth', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(f'{tier} sizes', fontsize=10)
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=7)

def make_id_figure(metric_key, ylabel, suptitle, save_as):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), squeeze=False)
    for ax, tier in zip(axes[0], TIERS):
        _plot_tier_panel(ax, tier, metric_key, ylabel)
    axes[0, -1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
                       fontsize=7.5, frameon=True)
    fig.suptitle(suptitle, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(save_as, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved -> {save_as}')

make_id_figure('pr',    'participation ratio',
               'Linear intrinsic dimensionality (PCA participation ratio) vs depth',
               'manifold_pr.png')
make_id_figure('twonn', 'TwoNN ID',
               'Non-linear intrinsic dimensionality (TwoNN) vs depth',
               'manifold_twonn.png')


---
## Local intrinsic dimensionality → video chunks

For each max-size FM, at its **last layer**, we compute the **local TwoNN ID** in each
window's k=`K_LOCAL` cosine neighborhood:

- **Low** local ID → the window sits in a *thin* region of the manifold — content that is locally repetitive / predictable.
- **High** local ID → the local neighborhood is *high-dimensional* — content that is locally unique / diverse.

Below: top-3 low and top-3 high windows per max-size FM, displayed as inline GIFs.
This is a manifold-geometry bridge to the video; different from the temporal-distance bridge in
`temporal_distance_analysis.ipynb`, which selected by *where neighbours live in time* rather than by
*local manifold shape*.


In [ ]:
def local_twonn(X, k=K_LOCAL):
    """Per-point local TwoNN ID. X (N, D) — L2-normalised internally."""
    Xn = _l2(X)
    nn = NearestNeighbors(n_neighbors=k + 1, metric='cosine').fit(Xn)
    _, idx = nn.kneighbors(Xn)
    out = np.empty(Xn.shape[0])
    for i in range(Xn.shape[0]):
        out[i] = twonn_id(Xn[idx[i, 1:]])  # exclude self
    return out

print('local_twonn() ready.')


In [ ]:
import cv2
from PIL import Image as PILImage
from IPython.display import HTML, display as ipy_display

CB_ROOT     = Path('.').resolve().parent
CLIP_CACHE  = CB_ROOT / '.clip_cache'
CLIP_MARKER = CLIP_CACHE / '.extracted'
_CLIP_INDEX = {}

def _build_clip_index(clips_dir):
    global _CLIP_INDEX
    _CLIP_INDEX.clear()
    for p in clips_dir.rglob('*.mp4'):
        digits = ''.join(c for c in p.stem if c.isdigit())
        if digits:
            _CLIP_INDEX[int(digits)] = p
    print(f'  Indexed {len(_CLIP_INDEX)} clips  '
          f'(range {min(_CLIP_INDEX)}..{max(_CLIP_INDEX)})')

def ensure_clips():
    if _CLIP_INDEX:
        return
    if CLIP_MARKER.exists():
        print(f'Using cached clips at {CLIP_CACHE}')
        _build_clip_index(CLIP_CACHE)
        return
    print('Downloading videos.tar (~2.6 GB) ...')
    tar_path = Path(hf_hub_download(repo_id=VIDEO_REPO, filename='videos.tar',
                                    repo_type='dataset', token=HF_TOKEN))
    CLIP_CACHE.mkdir(parents=True, exist_ok=True)
    import tarfile
    with tarfile.open(tar_path, 'r') as tar:
        tar.extractall(CLIP_CACHE)
    CLIP_MARKER.touch()
    _build_clip_index(CLIP_CACHE)

@functools.lru_cache(maxsize=512)
def _load_frames(clip_idx):
    cap = cv2.VideoCapture(str(_CLIP_INDEX[clip_idx]))
    frames = []
    while True:
        ok, f = cap.read()
        if not ok: break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return tuple(frames)

def get_window_frames(start_s, window_s, size=GIF_SIZE):
    end_s      = start_s + window_s
    first_clip = int(start_s // CLIP_SECONDS)
    last_clip  = min(int(np.ceil(end_s / CLIP_SECONDS)) - 1, MAX_CLIP_IDX)
    frames = []
    for c in range(first_clip, last_clip + 1):
        cf  = _load_frames(c)
        n   = len(cf)
        ct0 = c * CLIP_SECONDS
        f0  = max(0, int(np.floor((max(start_s, ct0) - ct0) / CLIP_SECONDS * n)))
        f1  = min(n, int(np.ceil((min(end_s, ct0 + CLIP_SECONDS) - ct0) / CLIP_SECONDS * n)))
        for frame in cf[f0:f1]:
            frames.append(cv2.resize(frame, (size, size), interpolation=cv2.INTER_LINEAR))
    return frames

def frames_to_gif(frames, fps=SOURCE_FPS):
    pil = [PILImage.fromarray(f) for f in frames]
    buf = io.BytesIO()
    pil[0].save(buf, format='GIF', save_all=True, append_images=pil[1:],
                duration=int(1000 / fps), loop=0, optimize=False)
    return buf.getvalue()

def gif_tag(frames, width=GIF_SIZE, border='none'):
    b64 = base64.b64encode(frames_to_gif(frames)).decode()
    return (f'<img src="data:image/gif;base64,{b64}" width="{width}" '
            f'style="border:{border}; image-rendering:auto;"/>')

print('Video helpers ready. Call ensure_clips() before visualising.')


In [ ]:
ensure_clips()

def _pick_extremes(scores, n=3, spacing=2):
    """Pick top-n low and top-n high indices, with min spacing between picks."""
    order_lo = np.argsort(scores)
    order_hi = order_lo[::-1]
    def take(order):
        picked = []
        for w in order:
            if all(abs(int(w) - p) > spacing for p in picked):
                picked.append(int(w))
                if len(picked) == n:
                    break
        return picked
    return take(order_lo), take(order_hi)

def _mmss(t):
    return f'{int(t // 60)}m{int(t % 60):02d}s'

def _window_seconds_for(fm):
    cfg = REGISTRY[fm]
    # EEG FMs: own window length. Vision FMs: family used = femba_luna → 5 s.
    return cfg['window_s'] if cfg['modality'] == 'eeg' else 5.0

LOCAL_ID_CACHE = {}   # fm -> np.ndarray of per-window local IDs

def extremes_section(fm):
    cfg  = REGISTRY[fm]
    size = cfg['tiers']['large']
    emb  = _load_emb(fm, size)            # (L, W, D)
    last = emb[-1]                        # (W, D)
    L_total, W, D = emb.shape
    print(f'\n=== {cfg["label"]}-{size}  last layer  W={W}  D={D} ===', flush=True)
    scores = local_twonn(last, k=K_LOCAL)
    LOCAL_ID_CACHE[fm] = scores
    lo_idx, hi_idx = _pick_extremes(scores)
    win_s = _window_seconds_for(fm)

    def _row_html(idxs, color, band_label):
        cells = []
        for w in idxs:
            t = float(w) * win_s
            frames = get_window_frames(t, win_s)
            gif = (gif_tag(frames, border=f'3px solid {color}')
                   if frames
                   else '<div style="width:160px;height:160px;background:#ccc;"></div>')
            cells.append(
                f'<td style="text-align:center;padding:4px;vertical-align:top;">'
                f'{gif}<br>'
                f'<span style="font-size:11px;color:{color};font-weight:bold;">'
                f'w={w}  {_mmss(t)}</span><br>'
                f'<span style="font-size:10px;">local ID={scores[w]:.2f}</span>'
                f'</td>')
        return (f'<tr><td style="font-weight:bold;color:{color};vertical-align:middle;'
                f'padding-right:8px;">{band_label}</td>{"".join(cells)}</tr>')

    html = (f'<h4 style="margin-bottom:4px;">{cfg["label"]}-{size}  &mdash; last layer '
            f'(L={L_total - 1})</h4>'
            f'<table style="border-collapse:collapse;">'
            f'{_row_html(lo_idx, "#1f77b4", "LOW")}'
            f'{_row_html(hi_idx, "#d62728", "HIGH")}'
            f'</table>'
            f'<p style="font-size:11px;color:#555;margin-top:2px;">'
            f'LOW = thin / predictable manifold neighborhood; HIGH = locally diverse.'
            f'</p>')
    ipy_display(HTML(html))

    # Static PNG companion (middle frame per window)
    fig, axes = plt.subplots(2, 3, figsize=(9, 6.4))
    for row, (idxs, color, label) in enumerate(
            [(lo_idx, '#1f77b4', 'LOW'), (hi_idx, '#d62728', 'HIGH')]):
        for col, w in enumerate(idxs):
            ax = axes[row, col]
            t = float(w) * win_s
            frames = get_window_frames(t, win_s)
            if frames:
                ax.imshow(frames[len(frames) // 2])
            else:
                ax.set_facecolor('#ccc')
            ax.set_title(f'{label}  w={w}  ID={scores[w]:.2f}',
                         fontsize=8, color=color, fontweight='bold', pad=3)
            ax.set_xlabel(_mmss(t), fontsize=7)
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(3)
            ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    fig.suptitle(f'{cfg["label"]}-{size}  last-layer local TwoNN extremes',
                 fontsize=10)
    plt.tight_layout()
    save_as = f'manifold_extremes_{fm}.png'
    plt.savefig(save_as, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  saved -> {save_as}')

for fm in REGISTRY.keys():
    extremes_section(fm)


---
## Cross-FM agreement on local manifold ID

If two FMs agree on **which windows are locally complex vs locally simple**, that is a
manifold-geometry "Platonic" signature distinct from the cross-modal mKNN alignment.
Below: Spearman correlation between per-window local TwoNN IDs across FMs that share
a window count (i.e. the same EEG family).


In [ ]:
from scipy.stats import spearmanr

groups = {}
for fm, scores in LOCAL_ID_CACHE.items():
    groups.setdefault(len(scores), []).append(fm)

print('FM groupings by window count:')
for w, fms in groups.items():
    print(f'  W={w}: {fms}')

for W, fms in groups.items():
    if len(fms) < 2:
        continue
    M = np.zeros((len(fms), len(fms)))
    for i, a in enumerate(fms):
        for j, b in enumerate(fms):
            if i == j:
                M[i, j] = 1.0
            else:
                rho, _ = spearmanr(LOCAL_ID_CACHE[a], LOCAL_ID_CACHE[b])
                M[i, j] = rho

    labels = [REGISTRY[f]['label'] for f in fms]
    fig, ax = plt.subplots(figsize=(max(4, len(fms) * 0.85 + 1.5),
                                     max(3.6, len(fms) * 0.75 + 1.5)))
    im = ax.imshow(M, vmin=-1, vmax=1, cmap='RdBu_r', origin='upper')
    ax.set_xticks(range(len(fms)))
    ax.set_yticks(range(len(fms)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(fms)):
        for j in range(len(fms)):
            ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center',
                    fontsize=7,
                    color='white' if abs(M[i, j]) > 0.5 else 'black')
    fig.colorbar(im, ax=ax, fraction=0.045, pad=0.04).set_label(
        'Spearman ρ', fontsize=9)
    ax.set_title(f'Cross-FM agreement on local TwoNN ID  (W={W})', fontsize=10)
    plt.tight_layout()
    save_as = f'manifold_localid_agreement_W{W}.png'
    plt.savefig(save_as, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  saved -> {save_as}')
